<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/06-numpy/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Exercises

Twelve exercises: four Easy, five Medium and three Hard. Each one is a small piece of real ML code written the NumPy way, with arrays instead of loops.

Each check compares your result against the expected array, allowing for tiny floating-point differences, and checks the shape and type too. Several also test the edge cases that NumPy makes easy to get wrong: in-place changes to the caller's array, negative indexes, overflow, and repeated indexes.

**Run the setup cell first.**

Rules:

- NumPy and the standard library only.
- Where an exercise says "no Python loops", it means no loop over samples or pixels. A loop that runs a handful of times, like once per batch, is fine.
- When something fails, print the shapes of your arrays. It's almost always a shape problem.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import time

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

_rng = np.random.default_rng(1234)


def check_standardize(f):
    X = np.array([[1, 10], [3, 30], [5, 50]])
    r = 1.0 / np.sqrt(2 / 3)

    def unchanged():
        X2 = np.array([[1.0, 2.0], [3.0, 5.0]])
        f(X2)
        return X2

    _scenarios("standardize", [
        ("basic", lambda: f(X), np.array([[-r, -r], [0.0, 0.0], [r, r]]),
         "subtract the column means and divide by the column stds (axis=0)"),
        ("result has float dtype", lambda: f(X).dtype.kind, "f", None),
        ("constant column becomes 0", lambda: f(np.array([[1.0, 7.0], [3.0, 7.0]])), np.array([[-1.0, 0.0], [1.0, 0.0]]),
         "where the std is 0, divide by 1 instead (np.where)"),
        ("input is not modified", unchanged, np.array([[1.0, 2.0], [3.0, 5.0]]), "don't use -= or /= on X itself"),
        ("each column ends up with mean 0 and std 1", lambda: (lambda Z: (Z.mean(axis=0), Z.std(axis=0)))(f(_rng.normal(5, 3, size=(200, 4)))),
         (np.zeros(4), np.ones(4)), None),
    ])


def check_one_hot(f):
    _scenarios("one_hot", [
        ("basic", lambda: f(np.array([0, 2, 1]), 3), np.array([[1.0, 0, 0], [0, 0, 1], [0, 1, 0]]), None),
        ("float dtype", lambda: f(np.array([1]), 2).dtype.kind, "f", None),
        ("no labels", lambda: f(np.array([], dtype=int), 3).shape, (0, 3), None),
        ("label too large", lambda: f(np.array([0, 3]), 3), Raises(ValueError), None),
        ("negative label", lambda: f(np.array([0, -1]), 3), Raises(ValueError),
         "np.eye(3)[-1] silently picks the last row. Check for negatives explicitly"),
    ])


def check_softmax(f):
    def unchanged():
        z = np.array([1.0, 2.0, 3.0])
        f(z)
        return z

    _scenarios("softmax", [
        ("1D", lambda: f(np.array([1.0, 2.0, 3.0])), np.array([0.09003057, 0.24472847, 0.66524096]), None),
        ("2D: one distribution per row", lambda: f(np.array([[0.0, 0.0], [0.0, np.log(3.0)]])), np.array([[0.5, 0.5], [0.25, 0.75]]),
         "work along the last axis, with keepdims=True"),
        ("rows sum to 1", lambda: f(_rng.normal(size=(5, 7))).sum(axis=1), np.ones(5), None),
        ("large values don't overflow", lambda: f(np.array([1000.0, 1001.0])), np.array([0.26894142, 0.73105858]),
         "subtract the max of each row before calling np.exp"),
        ("input is not modified", unchanged, np.array([1.0, 2.0, 3.0]), "z -= ... changes the caller's array; use z - ... instead"),
    ])


def check_split_indices(f):
    def expected(n, ratio, seed):
        perm = np.random.default_rng(seed).permutation(n)
        k = round(n * ratio)
        return perm[k:], perm[:k]

    def covers():
        tr, te = f(50, 0.2, 3)
        return (len(tr), len(te), sorted(np.concatenate([tr, te]).tolist()) == list(range(50)))

    _scenarios("split_indices", [
        ("follows the rule exactly", lambda: f(10, 0.3, 0), expected(10, 0.3, 0),
         "perm = np.random.default_rng(seed).permutation(n); test is the first round(n * ratio)"),
        ("sizes and coverage", covers, (40, 10, True), None),
        ("same seed, same split", lambda: _same(f(30, 0.2, 7), f(30, 0.2, 7)), True, None),
        ("ratio 0", lambda: len(f(5, 0.0, 1)[1]), 0, None),
        ("integer indexes", lambda: f(5, 0.4, 1)[0].dtype.kind, "i", None),
    ])


def _naive_dists(A, B):
    return np.array([[np.sqrt(((a - b) ** 2).sum()) for b in B] for a in A])


def check_pairwise_distances(f):
    A = _rng.normal(size=(6, 3))
    B = _rng.normal(size=(4, 3))

    def no_nan():
        P = _rng.normal(1000, 1, size=(50, 5))
        with np.errstate(invalid="ignore"):
            D = f(P, P)
        return (bool(np.isnan(D).any()), bool((D < 0).any()))

    _scenarios("pairwise_distances", [
        ("matches the direct formula", lambda: f(A, B), _naive_dists(A, B), None),
        ("shape (n, m)", lambda: f(A, B).shape, (6, 4), None),
        ("simple case", lambda: f(np.array([[0.0, 0.0]]), np.array([[3.0, 4.0], [0.0, 1.0]])), np.array([[5.0, 1.0]]), None),
        ("no NaN or negative values for identical points", no_nan, (False, False),
         "rounding can make a squared distance like -1e-9; clip at 0 before np.sqrt"),
    ])


def check_knn_predict(f):
    X_train = np.array([[0.0], [1.0], [2.0], [10.0], [11.0], [12.0]])
    y_train = np.array([0, 0, 0, 1, 1, 1])

    def against_naive():
        Xtr = _rng.normal(size=(60, 3))
        ytr = _rng.integers(0, 3, size=60)
        Xte = _rng.normal(size=(15, 3))
        D = _naive_dists(Xte, Xtr)
        expected = []
        for row in D:
            labels = ytr[np.argsort(row, kind="stable")[:5]]
            counts = np.bincount(labels, minlength=3)
            expected.append(int(np.argmax(counts)))
        return _same(f(Xtr, ytr, Xte, 5), np.array(expected))

    _scenarios("knn_predict", [
        ("simple", lambda: f(X_train, y_train, np.array([[1.5], [10.5]]), 3), np.array([0, 1]), None),
        ("k = 1", lambda: f(X_train, y_train, np.array([[5.0], [7.0]]), 1), np.array([0, 1]), None),
        ("a tie goes to the smaller label", lambda: f(np.array([[0.0], [2.0]]), np.array([1, 0]), np.array([[1.0]]), 2), np.array([0]),
         "count votes per class and take argmax, which picks the smallest label on a tie"),
        ("matches a direct implementation", against_naive, True,
         "count the votes for each class (np.add.at or np.bincount) and take argmax; don't average the labels"),
    ])


def check_confusion_matrix(f):
    _scenarios("confusion_matrix", [
        ("basic", lambda: f(np.array([0, 0, 1, 2, 1, 0]), np.array([0, 1, 1, 2, 0, 0]), 3),
         np.array([[2, 1, 0], [1, 1, 0], [0, 0, 1]]),
         "rows are true labels, columns are predictions. If some counts are too low, M[t, p] += 1 is ignoring repeated pairs: use np.add.at or np.bincount"),
        ("integer dtype", lambda: f(np.array([0]), np.array([0]), 2).dtype.kind, "i", None),
        ("classes that never appear", lambda: f(np.array([1]), np.array([1]), 4).shape, (4, 4), None),
        ("repeated pairs add up", lambda: f(np.array([1, 1, 1]), np.array([0, 0, 0]), 2), np.array([[0, 0], [3, 0]]),
         "M[t, p] += 1 with fancy indexing only adds 1 once for repeated pairs. Use np.add.at or np.bincount"),
        ("label out of range", lambda: f(np.array([0, 3]), np.array([0, 0]), 3), Raises(ValueError), None),
    ])


def check_moving_average(f):
    x = _rng.normal(size=200)
    naive = np.array([x[i:i + 7].mean() for i in range(len(x) - 6)])
    _scenarios("moving_average", [
        ("small example", lambda: f(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), 2), np.array([1.5, 2.5, 3.5, 4.5]),
         "prepend a 0 to the cumulative sum so the first window works: c[w:] - c[:-w]"),
        ("window = length", lambda: f(np.array([2.0, 4.0]), 2), np.array([3.0]), None),
        ("matches the direct calculation", lambda: f(x, 7), naive,
         "prepend a 0 to the cumulative sum so the first window works: c[w:] - c[:-w]"),
        ("window 0", lambda: f(np.array([1.0, 2.0]), 0), Raises(ValueError), None),
        ("window too big", lambda: f(np.array([1.0, 2.0]), 3), Raises(ValueError), None),
    ])


def check_minibatches(f):
    X = np.arange(20).reshape(10, 2)
    y = np.arange(10)

    def in_order():
        return [(xb[:, 0].tolist(), yb.tolist()) for xb, yb in f(X, y, 4)]

    def shuffled_ok():
        batches = list(f(X, y, 3, shuffle=True, seed=0))
        rows = np.concatenate([yb for _, yb in batches])
        aligned = all((xb[:, 0] == 2 * yb).all() for xb, yb in batches)
        return (sorted(rows.tolist()) == list(range(10)), rows.tolist() != list(range(10)), aligned)

    def reproducible():
        a = [yb.tolist() for _, yb in f(X, y, 3, shuffle=True, seed=5)]
        b = [yb.tolist() for _, yb in f(X, y, 3, shuffle=True, seed=5)]
        return a == b

    _scenarios("minibatches", [
        ("is a generator", lambda: inspect.isgenerator(f(X, y, 2)), True, "use yield"),
        ("in order, last batch smaller", in_order,
         [([0, 2, 4, 6], [0, 1, 2, 3]), ([8, 10, 12, 14], [4, 5, 6, 7]), ([16, 18], [8, 9])], None),
        ("shuffled: every row once, X and y still matched", shuffled_ok, (True, True, True),
         "shuffle one array of indexes and use it for both X and y"),
        ("same seed, same order", reproducible, True, None),
        ("X and y of different lengths", lambda: next(f(X, y[:5], 2)), Raises(ValueError), None),
    ])


def check_ridge_fit(f):
    xs = np.linspace(0, 5, 20)
    X1 = xs[:, None]
    y1 = 2 * xs + 1

    def multi():
        X = _rng.normal(size=(100, 3))
        y = X @ np.array([1.5, -2.0, 0.5]) + 4.0
        w, b = f(X, y)
        return (w, b)

    def shrinks():
        X = _rng.normal(size=(50, 2))
        y = X @ np.array([3.0, -1.0]) + _rng.normal(0, 0.1, size=50)
        w0, _ = f(X, y, 0.0)
        w1, _ = f(X, y, 100.0)
        return bool(np.linalg.norm(w1) < np.linalg.norm(w0))

    def bias_not_penalised():
        X = _rng.normal(size=(200, 1))
        y = 0.0 * X[:, 0] + 50.0
        _, b = f(X, y, 1000.0)
        return b

    _scenarios("ridge_fit", [
        ("exact line, no regularization", lambda: f(X1, y1), (np.array([2.0]), 1.0), None),
        ("several features", multi, (np.array([1.5, -2.0, 0.5]), 4.0), None),
        ("l2 shrinks the weights", shrinks, True, None),
        ("the bias isn't regularized", bias_not_penalised, 50.0,
         "center X and y first, solve for w, then b = mean(y) - mean(X) @ w"),
    ])


def check_cross_entropy_with_grad(f):
    def numerical():
        logits = _rng.normal(size=(4, 3))
        y = np.array([0, 2, 1, 2])
        _, grad = f(logits, y)
        num = np.zeros_like(logits)
        h = 1e-6
        for i in range(4):
            for j in range(3):
                plus, minus = logits.copy(), logits.copy()
                plus[i, j] += h
                minus[i, j] -= h
                num[i, j] = (f(plus, y)[0] - f(minus, y)[0]) / (2 * h)
        return bool(np.allclose(grad, num, atol=1e-6))

    _scenarios("cross_entropy_with_grad", [
        ("uniform logits give log(k)", lambda: f(np.zeros((2, 4)), np.array([0, 3]))[0], float(np.log(4)), None),
        ("a confident correct prediction has a small loss", lambda: f(np.array([[10.0, 0.0]]), np.array([0]))[0] < 1e-3, True, None),
        ("large logits stay finite", lambda: np.isfinite(f(np.array([[1000.0, -1000.0]]), np.array([1]))[0]), True,
         "use log-softmax: z - max - log(sum(exp(z - max))), never log(softmax(z))"),
        ("gradient rows sum to 0", lambda: f(_rng.normal(size=(5, 3)), np.array([0, 1, 2, 0, 1]))[1].sum(axis=1), np.zeros(5), None),
        ("gradient matches a numerical estimate", numerical, True, "grad = (softmax(logits) - one_hot(y)) / n"),
    ])


def check_conv2d(f):
    def naive(img, ker):
        kh, kw = ker.shape
        H, W = img.shape
        out = np.zeros((H - kh + 1, W - kw + 1))
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = (img[i:i + kh, j:j + kw] * ker).sum()
        return out

    img = _rng.normal(size=(9, 12))
    ker = _rng.normal(size=(3, 4))
    edge = np.array([[0, 0, 1, 1]] * 4, dtype=float)
    _scenarios("conv2d", [
        ("matches the direct calculation", lambda: f(img, ker), naive(img, ker),
         "out[i, j] = (image[i:i+kh, j:j+kw] * kernel).sum(), with the kernel not flipped"),
        ("output shape", lambda: f(np.zeros((10, 8)), np.zeros((3, 3))).shape, (8, 6), None),
        ("vertical edge detector", lambda: f(edge, np.array([[-1.0, 1.0]])), np.array([[0.0, 1.0, 0.0]] * 4),
         "this is cross-correlation: don't flip the kernel"),
        ("kernel bigger than the image", lambda: f(np.zeros((2, 2)), np.zeros((3, 1))), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `standardize`

Write `standardize(X)` that takes a 2D array of shape `(n_samples, n_features)` and returns a new float array where every column has mean 0 and standard deviation 1.

- If a column is constant (standard deviation 0), it should become all zeros instead of causing a division by zero.
- Don't modify `X`.
- No Python loops: use `axis` and broadcasting.

In [ ]:
import numpy as np


def standardize(X: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_standardize(standardize)

### Exercise 2: `one_hot`

Write `one_hot(labels, n_classes)` that turns an integer array of labels of shape `(n,)` into a float array of shape `(n, n_classes)`, with a 1 in each row at the label's position.

```python
one_hot(np.array([0, 2, 1]), 3)
# [[1, 0, 0],
#  [0, 0, 1],
#  [0, 1, 0]]
```

Raise a `ValueError` if any label is negative or not less than `n_classes`. No Python loops.

In [ ]:
import numpy as np


def one_hot(labels: np.ndarray, n_classes: int) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_one_hot(one_hot)

### Exercise 3: `softmax`

Write `softmax(z)` that turns scores into probabilities along the **last axis**, so it works for a single vector `(k,)` and for a batch `(n, k)`.

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

- It must not overflow for large values like `[1000, 1001]`.
- Don't modify the input.
- No Python loops.

In [ ]:
import numpy as np


def softmax(z: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_softmax(softmax)

### Exercise 4: `split_indices`

Write `split_indices(n, test_ratio, seed)` that returns `(train_idx, test_idx)`: two integer arrays that split the indexes `0` to `n - 1` into a training part and a test part.

Use exactly this rule, so the result is reproducible:

1. `perm = np.random.default_rng(seed).permutation(n)`
2. The first `round(n * test_ratio)` entries of `perm` are the test indexes, and the rest are the training indexes.

You'd then use these with fancy indexing: `X[train_idx], y[train_idx]`.

In [ ]:
import numpy as np


def split_indices(n: int, test_ratio: float, seed: int):
    # your code here
    pass

In [ ]:
check_split_indices(split_indices)

## Medium

### Exercise 5: `pairwise_distances`

Write `pairwise_distances(A, B)` that returns the Euclidean distance between every row of `A` (shape `(n, d)`) and every row of `B` (shape `(m, d)`), as an array of shape `(n, m)`.

No Python loops. You can use broadcasting with an extra axis, as in section 6 of the lesson, or the expansion from section 10:

$$\|a - b\|^2 = \|a\|^2 - 2\, a \cdot b + \|b\|^2$$

Make sure the result never contains NaN or negative values, even for identical points.

In [ ]:
import numpy as np


def pairwise_distances(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_pairwise_distances(pairwise_distances)

### Exercise 6: `knn_predict`

Write `knn_predict(X_train, y_train, X_test, k)` that predicts a label for every test point with k-nearest neighbours.

- Labels are integers from 0 upwards.
- If there's a tie in the vote, predict the **smallest** label.
- No Python loops over the test points. (You can use your `pairwise_distances` from exercise 5.)

```python
knn_predict(X_train, y_train, X_test, k=5)   # array of shape (n_test,)
```

In [ ]:
import numpy as np


def knn_predict(X_train, y_train, X_test, k: int) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_knn_predict(knn_predict)

### Exercise 7: `confusion_matrix`

Write `confusion_matrix(y_true, y_pred, n_classes)` that returns an integer array `M` of shape `(n_classes, n_classes)`, where `M[t, p]` counts the samples with true label `t` predicted as `p`.

- Raise a `ValueError` if any label is outside `0` to `n_classes - 1`.
- No Python loops over the samples.

Careful: the obvious one-liner, `M[y_true, y_pred] += 1`, has a subtle bug. See if you can spot it, or let the check find it.

In [ ]:
import numpy as np


def confusion_matrix(y_true, y_pred, n_classes: int) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_confusion_matrix(confusion_matrix)

### Exercise 8: `moving_average`

Write `moving_average(x, window)` that returns the mean of every run of `window` consecutive values, as a float array of length `len(x) - window + 1`.

Use a **cumulative sum** (`np.cumsum`) so it takes the same time whatever the window size, with no Python loops.

Raise a `ValueError` if `window` is less than 1 or greater than `len(x)`.

You wrote this with a running total in Module 2. This is the NumPy version of the same idea.

In [ ]:
import numpy as np


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_moving_average(moving_average)

### Exercise 9: `minibatches`

Write a **generator** `minibatches(X, y, batch_size, shuffle=False, seed=None)` that yields `(X_batch, y_batch)` pairs.

- Without shuffling, batches come in order. The last batch can be smaller.
- With `shuffle=True`, the order is random (using `np.random.default_rng(seed)`), every row appears exactly once, and each `X` row stays with its own `y` value.
- Raise a `ValueError` if `X` and `y` have different lengths.

This is the NumPy version of the `DataLoader` from Module 3.

In [ ]:
import numpy as np


def minibatches(X, y, batch_size: int, shuffle: bool = False, seed=None):
    # your code here
    pass

In [ ]:
check_minibatches(minibatches)

## Hard

### Exercise 10: `ridge_fit`

Write `ridge_fit(X, y, l2=0.0)` that fits a linear model `y ≈ X @ w + b` by solving the equations directly, and returns `(w, b)`.

**Ridge regression** adds a penalty on large weights, controlled by `l2`. The bias `b` is **not** penalised. The standard way to do that is:

1. Center the data: subtract the column means from `X` and the mean from `y`.
2. Solve $(X_c^\top X_c + \lambda I)\, w = X_c^\top y_c$ for `w`, using `np.linalg.solve`.
3. Recover the bias: `b = mean(y) - mean(X) @ w`.

With `l2=0` this is ordinary least squares.

In [ ]:
import numpy as np


def ridge_fit(X: np.ndarray, y: np.ndarray, l2: float = 0.0):
    # your code here
    pass

In [ ]:
check_ridge_fit(ridge_fit)

### Exercise 11: `cross_entropy_with_grad`

The loss function for almost every classifier, from logistic regression to large language models, is **cross-entropy**. Given raw scores (*logits*) of shape `(n, k)` and integer labels `y` of shape `(n,)`, it's the average of minus the log of the probability assigned to the correct class:

$$L = -\frac{1}{n} \sum_i \log \text{softmax}(z_i)_{y_i}$$

Write `cross_entropy_with_grad(logits, y)` that returns `(loss, grad)`, where `grad` is the gradient of the loss with respect to the logits, with the same shape as the logits. It has a famously simple form:

$$\frac{\partial L}{\partial z} = \frac{1}{n}\left(\text{softmax}(z) - \text{one\_hot}(y)\right)$$

- The loss must stay finite for very large logits like `[[1000, -1000]]`.
- No Python loops.

In [ ]:
import numpy as np


def cross_entropy_with_grad(logits: np.ndarray, y: np.ndarray):
    # your code here
    pass

In [ ]:
check_cross_entropy_with_grad(cross_entropy_with_grad)

### Exercise 12: `conv2d`

Convolutional neural networks (CNNs), the classic architecture for images, are built on one operation: slide a small grid of weights (the *kernel*) over the image, and at each position, multiply the overlapping values and add them up.

Write `conv2d(image, kernel)` for a 2D image of shape `(H, W)` and a kernel of shape `(kh, kw)`:

- Return an array of shape `(H - kh + 1, W - kw + 1)`, where `out[i, j] = (image[i:i+kh, j:j+kw] * kernel).sum()`. (This is what deep learning libraries call convolution. Mathematicians call it *cross-correlation*, since the kernel isn't flipped.)
- Raise a `ValueError` if the kernel is bigger than the image in either direction.
- Don't loop over output pixels in Python. Two options: loop over the kernel's positions instead (only `kh * kw` iterations), or use `np.lib.stride_tricks.sliding_window_view`.

In [ ]:
import numpy as np


def conv2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [ ]:
check_conv2d(conv2d)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where your `softmax`, `cross_entropy_with_grad` and `minibatches` come together in a neural network that reads handwritten digits.